# Model 1: Watershed Segmentation (Classical CV)\n\nClassical computer vision approach for waste segmentation and counting.\n\n**Pipeline**: Preprocessing → Color filtering → Morphology → Distance Transform → Watershed → Counting\n\n**Strengths**: Fast inference, no GPU needed, interpretable\n**Limitations**: Cannot classify waste categories, struggles with cluttered scenes\n\nThis serves as the **classical baseline** to compare against deep learning models.

In [ ]:
import cv2
import numpy as np
import json
import os
import time
from collections import defaultdict
import matplotlib.pyplot as plt
from scipy import ndimage

# =========================
# PATHS
# =========================

BASE_DIR = r"C:\Users\User\Desktop\Ipynb"
DATASET_DIR = os.path.join(BASE_DIR, "archive", "dataset_v2")
OUTPUT_DIR = os.path.join(BASE_DIR, "runs", "watershed")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load test annotations for evaluation
with open(os.path.join(DATASET_DIR, "test_annotations.json")) as f:
    test_data = json.load(f)

with open(os.path.join(DATASET_DIR, "val_annotations.json")) as f:
    val_data = json.load(f)

print(f"Val images: {len(val_data['images'])}")
print(f"Test images: {len(test_data['images'])}")

## Watershed Segmentation Pipeline\n\nThe core algorithm:\n1. Convert to LAB color space (better color separation)\n2. Apply bilateral filtering (smooth while preserving edges)\n3. Multi-channel thresholding (Otsu on L + adaptive on A/B channels)\n4. Morphological cleanup (opening to remove noise, closing to fill gaps)\n5. Distance transform to find object centers\n6. Marker-based watershed to separate touching objects\n7. Filter by minimum area and extract contours

In [ ]:
def watershed_segment(image, params=None):
    """
    Perform watershed segmentation on a waste image.
    
    Args:
        image: BGR image (numpy array)
        params: dict of hyperparameters
    
    Returns:
        labels: labeled mask where each object has a unique ID
        num_objects: number of detected objects
        contours: list of contours for each object
        binary_mask: binary mask (waste=255, background=0)
    """
    if params is None:
        params = {
            "blur_d": 9,              # bilateral filter diameter
            "blur_sigma_color": 75,    # bilateral filter sigma color
            "blur_sigma_space": 75,    # bilateral filter sigma space
            "morph_kernel": 5,         # morphological kernel size
            "dist_threshold": 0.4,     # distance transform threshold (0-1)
            "min_area": 500,           # minimum contour area in pixels
            "edge_weight": 0.3,        # weight for edge-based detection
        }
    
    h, w = image.shape[:2]
    
    # 1. Bilateral filter (smooth while preserving edges)
    filtered = cv2.bilateralFilter(
        image, params["blur_d"], 
        params["blur_sigma_color"], 
        params["blur_sigma_space"]
    )
    
    # 2. Convert to multiple color spaces
    lab = cv2.cvtColor(filtered, cv2.COLOR_BGR2LAB)
    hsv = cv2.cvtColor(filtered, cv2.COLOR_BGR2HSV)
    gray = cv2.cvtColor(filtered, cv2.COLOR_BGR2GRAY)
    
    # 3. Multi-channel thresholding
    l_chan, a_chan, b_chan = cv2.split(lab)
    
    # Otsu on L channel (brightness)
    _, thresh_l = cv2.threshold(l_chan, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    
    # Adaptive threshold on A channel (green-red axis - good for colored waste)
    thresh_a = cv2.adaptiveThreshold(
        a_chan, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
        cv2.THRESH_BINARY, 21, 2
    )
    
    # Edge detection (Canny) to capture object boundaries
    edges = cv2.Canny(gray, 50, 150)
    edges_dilated = cv2.dilate(edges, np.ones((3, 3), np.uint8), iterations=1)
    
    # Saturation channel (waste is often more colorful than background)
    _, thresh_sat = cv2.threshold(hsv[:, :, 1], 40, 255, cv2.THRESH_BINARY)
    
    # Combine thresholds
    combined = cv2.bitwise_or(thresh_l, thresh_a)
    combined = cv2.bitwise_or(combined, thresh_sat)
    edge_contribution = (edges_dilated * params["edge_weight"]).astype(np.uint8)
    combined = cv2.add(combined, edge_contribution)
    _, combined = cv2.threshold(combined, 127, 255, cv2.THRESH_BINARY)
    
    # 4. Morphological cleanup
    kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE, 
        (params["morph_kernel"], params["morph_kernel"])
    )
    # Opening: remove small noise
    cleaned = cv2.morphologyEx(combined, cv2.MORPH_OPEN, kernel, iterations=2)
    # Closing: fill small holes
    cleaned = cv2.morphologyEx(cleaned, cv2.MORPH_CLOSE, kernel, iterations=2)
    
    binary_mask = cleaned.copy()
    
    # 5. Sure background (dilate to expand background region)
    sure_bg = cv2.dilate(cleaned, kernel, iterations=3)
    
    # 6. Distance transform for sure foreground
    dist_transform = cv2.distanceTransform(cleaned, cv2.DIST_L2, 5)
    dist_threshold = params["dist_threshold"] * dist_transform.max()
    _, sure_fg = cv2.threshold(dist_transform, dist_threshold, 255, 0)
    sure_fg = np.uint8(sure_fg)
    
    # 7. Unknown region
    unknown = cv2.subtract(sure_bg, sure_fg)
    
    # 8. Label markers for watershed
    num_labels, markers = cv2.connectedComponents(sure_fg)
    markers = markers + 1  # background should not be 0
    markers[unknown == 255] = 0  # unknown region marked as 0
    
    # 9. Apply watershed
    markers = cv2.watershed(image, markers)
    
    # 10. Extract results
    contours_list = []
    valid_labels = set(np.unique(markers)) - {-1, 1}  # exclude boundary (-1) and background (1)
    
    object_count = 0
    result_mask = np.zeros((h, w), dtype=np.int32)
    
    for label_id in valid_labels:
        region_mask = np.uint8(markers == label_id) * 255
        area = cv2.countNonZero(region_mask)
        
        if area < params["min_area"]:
            continue
        
        contours, _ = cv2.findContours(region_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if contours:
            object_count += 1
            result_mask[markers == label_id] = object_count
            contours_list.append(contours[0])
    
    return result_mask, object_count, contours_list, binary_mask


print("Watershed pipeline defined.")

## Hyperparameter Tuning on Validation Set\n\nGrid search over key parameters using counting MAE as the metric.

In [ ]:
# =========================
# BUILD GROUND TRUTH COUNTS
# =========================

def get_gt_counts(data):
    """Get ground truth object count per image."""
    ann_by_img = defaultdict(int)
    for ann in data["annotations"]:
        ann_by_img[ann["image_id"]] += 1
    
    gt = {}
    for img in data["images"]:
        gt[img["file_name"]] = ann_by_img.get(img["id"], 0)
    return gt

def get_gt_masks(data, image_shape):
    """Get ground truth binary mask for an image from COCO annotations."""
    masks = []
    for ann in data:
        if "segmentation" not in ann:
            continue
        for seg in ann["segmentation"]:
            pts = np.array(seg, dtype=np.float32).reshape(-1, 2)
            pts = pts.astype(np.int32)
            mask = np.zeros(image_shape[:2], dtype=np.uint8)
            cv2.fillPoly(mask, [pts], 255)
            masks.append(mask)
    
    if masks:
        combined = np.zeros(image_shape[:2], dtype=np.uint8)
        for m in masks:
            combined = cv2.bitwise_or(combined, m)
        return combined
    return np.zeros(image_shape[:2], dtype=np.uint8)

val_gt_counts = get_gt_counts(val_data)
val_ann_by_image = defaultdict(list)
for ann in val_data["annotations"]:
    val_ann_by_image[ann["image_id"]].append(ann)

val_image_lookup = {img["file_name"]: img for img in val_data["images"]}

print(f"Val ground truth: {len(val_gt_counts)} images")
print(f"Total val objects: {sum(val_gt_counts.values())}")

In [ ]:
# =========================
# GRID SEARCH ON VALIDATION SET
# =========================

param_grid = [
    {"blur_d": 9, "blur_sigma_color": 75, "blur_sigma_space": 75,
     "morph_kernel": 5, "dist_threshold": 0.3, "min_area": 300, "edge_weight": 0.3},
    {"blur_d": 9, "blur_sigma_color": 75, "blur_sigma_space": 75,
     "morph_kernel": 5, "dist_threshold": 0.4, "min_area": 500, "edge_weight": 0.3},
    {"blur_d": 9, "blur_sigma_color": 75, "blur_sigma_space": 75,
     "morph_kernel": 7, "dist_threshold": 0.5, "min_area": 500, "edge_weight": 0.2},
    {"blur_d": 11, "blur_sigma_color": 100, "blur_sigma_space": 100,
     "morph_kernel": 7, "dist_threshold": 0.4, "min_area": 800, "edge_weight": 0.3},
    {"blur_d": 7, "blur_sigma_color": 50, "blur_sigma_space": 50,
     "morph_kernel": 3, "dist_threshold": 0.3, "min_area": 200, "edge_weight": 0.4},
]

val_image_dir = os.path.join(DATASET_DIR, "images", "val")
val_image_files = [f for f in os.listdir(val_image_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

print(f"Tuning on {len(val_image_files)} validation images with {len(param_grid)} parameter sets...\n")

best_params = None
best_mae = float("inf")

for pidx, params in enumerate(param_grid):
    errors = []
    ious = []
    
    for fname in val_image_files:
        img_path = os.path.join(val_image_dir, fname)
        image = cv2.imread(img_path)
        if image is None:
            continue
        
        gt_count = val_gt_counts.get(fname, 0)
        
        _, pred_count, _, binary_mask = watershed_segment(image, params)
        
        errors.append(abs(pred_count - gt_count))
        
        # Binary IoU
        img_info = val_image_lookup.get(fname)
        if img_info:
            img_id = img_info["id"]
            gt_mask = get_gt_masks(val_ann_by_image.get(img_id, []), image.shape)
            
            intersection = cv2.bitwise_and(binary_mask, gt_mask)
            union = cv2.bitwise_or(binary_mask, gt_mask)
            
            inter_area = cv2.countNonZero(intersection)
            union_area = cv2.countNonZero(union)
            
            iou = inter_area / union_area if union_area > 0 else 0.0
            ious.append(iou)
    
    mae = np.mean(errors)
    mean_iou = np.mean(ious) if ious else 0.0
    
    print(f"  Params {pidx}: MAE={mae:.2f}, Binary IoU={mean_iou:.3f}, "
          f"dist_thresh={params['dist_threshold']}, min_area={params['min_area']}, "
          f"morph_k={params['morph_kernel']}")
    
    if mae < best_mae:
        best_mae = mae
        best_params = params

print(f"\nBest params: MAE={best_mae:.2f}")
print(f"  {best_params}")

## Evaluate on Test Set with Best Parameters

In [ ]:
# =========================
# EVALUATE ON TEST SET
# =========================

test_gt_counts = get_gt_counts(test_data)
test_ann_by_image = defaultdict(list)
for ann in test_data["annotations"]:
    test_ann_by_image[ann["image_id"]].append(ann)
test_image_lookup = {img["file_name"]: img for img in test_data["images"]}

test_image_dir = os.path.join(DATASET_DIR, "images", "test")
test_image_files = [f for f in os.listdir(test_image_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

print(f"Evaluating on {len(test_image_files)} test images with best params...\n")

results = []
inference_times = []

for fname in test_image_files:
    img_path = os.path.join(test_image_dir, fname)
    image = cv2.imread(img_path)
    if image is None:
        continue
    
    gt_count = test_gt_counts.get(fname, 0)
    
    # Measure inference time
    start = time.time()
    labels, pred_count, contours, binary_mask = watershed_segment(image, best_params)
    elapsed = (time.time() - start) * 1000  # ms
    inference_times.append(elapsed)
    
    # Counting error
    count_error = abs(pred_count - gt_count)
    
    # Binary IoU
    img_info = test_image_lookup.get(fname)
    iou = 0.0
    if img_info:
        img_id = img_info["id"]
        gt_mask = get_gt_masks(test_ann_by_image.get(img_id, []), image.shape)
        intersection = cv2.bitwise_and(binary_mask, gt_mask)
        union = cv2.bitwise_or(binary_mask, gt_mask)
        inter_area = cv2.countNonZero(intersection)
        union_area = cv2.countNonZero(union)
        iou = inter_area / union_area if union_area > 0 else 0.0
    
    results.append({
        "file": fname,
        "gt_count": gt_count,
        "pred_count": pred_count,
        "count_error": count_error,
        "binary_iou": iou,
        "inference_ms": elapsed,
    })

# Compute summary metrics
count_errors = [r["count_error"] for r in results]
binary_ious = [r["binary_iou"] for r in results]
within_1 = sum(1 for e in count_errors if e <= 1) / len(count_errors) * 100
within_3 = sum(1 for e in count_errors if e <= 3) / len(count_errors) * 100

print("=" * 50)
print("WATERSHED SEGMENTATION - TEST RESULTS")
print("=" * 50)
print(f"Counting MAE:           {np.mean(count_errors):.2f}")
print(f"Counting within ±1:     {within_1:.1f}%")
print(f"Counting within ±3:     {within_3:.1f}%")
print(f"Binary Seg IoU:         {np.mean(binary_ious):.3f}")
print(f"Avg inference time:     {np.mean(inference_times):.1f} ms")
print(f"Median inference time:  {np.median(inference_times):.1f} ms")
print("=" * 50)

# Save results
import json as json_module
results_path = os.path.join(OUTPUT_DIR, "watershed_results.json")
with open(results_path, "w") as f:
    json_module.dump({
        "metrics": {
            "counting_mae": float(np.mean(count_errors)),
            "counting_within_1": float(within_1),
            "counting_within_3": float(within_3),
            "binary_iou": float(np.mean(binary_ious)),
            "avg_inference_ms": float(np.mean(inference_times)),
        },
        "best_params": best_params,
        "per_image": results,
    }, f, indent=2)
print(f"\nResults saved to: {results_path}")

## Visualize Sample Results

In [ ]:
# =========================
# VISUALIZE SAMPLE RESULTS
# =========================

# Pick 5 best and 5 worst by counting error
sorted_results = sorted(results, key=lambda r: r["count_error"])
samples = sorted_results[:5] + sorted_results[-5:]

fig, axes = plt.subplots(len(samples), 3, figsize=(18, 5 * len(samples)))

for i, r in enumerate(samples):
    img_path = os.path.join(test_image_dir, r["file"])
    image = cv2.imread(img_path)
    if image is None:
        continue
    
    labels, pred_count, contours, binary_mask = watershed_segment(image, best_params)
    
    # Original
    axes[i, 0].imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    axes[i, 0].set_title(f"Original (GT: {r['gt_count']} objects)")
    axes[i, 0].axis("off")
    
    # Binary mask
    axes[i, 1].imshow(binary_mask, cmap="gray")
    axes[i, 1].set_title(f"Binary Mask (IoU: {r['binary_iou']:.3f})")
    axes[i, 1].axis("off")
    
    # Watershed result with contours
    overlay = image.copy()
    cv2.drawContours(overlay, contours, -1, (0, 255, 0), 2)
    
    # Color each segment differently
    colored_labels = np.zeros_like(image)
    for label_id in range(1, labels.max() + 1):
        color = np.random.randint(50, 255, 3).tolist()
        colored_labels[labels == label_id] = color
    
    blended = cv2.addWeighted(image, 0.6, colored_labels, 0.4, 0)
    axes[i, 2].imshow(cv2.cvtColor(blended, cv2.COLOR_BGR2RGB))
    axes[i, 2].set_title(f"Watershed (Pred: {pred_count}, Error: {r['count_error']})")
    axes[i, 2].axis("off")

plt.suptitle("Watershed Results: Top 5 Best (top) + Top 5 Worst (bottom)", fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "watershed_visualization.png"), dpi=100, bbox_inches="tight")
plt.show()
print("Visualization saved.")